# Modul B · Kapitel 2.4 — Augmented Prompt

## Challenge: Aus Frage und Chunks wird ein Prompt, der Belege liefert


**Lernziel:** Du kannst aus einer Frage und den Treffern einer Suche einen Prompt bauen, der
belegte Antworten erzeugt — mit Quellenangaben, mit einem gerechneten Token-Budget und in einer
Reihenfolge, die zur Aufmerksamkeit des Modells passt.

Dieses Notebook baut das rechte Ende der RAG-Kette:

```
Dokumente ──► Chunks ──► Embeddings ──► Vector Database ──► Retrieval ──► Prompt ──► Antwort
                                                                          └─ hier ─┘
```

Die Suche liefert rund zwanzig Kandidaten. Zwischen ihnen und dem Modell steht die Arbeit dieses
Notebooks: eine Instruktion, ein Re-Ranking mit Entdopplung und Token-Budget, ein Kontextteil mit
Quellenangaben und ein Antwortformat.

### So funktioniert dieses Notebook

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Erklärung — lesen |
| ▶️ | Fertiger Code — einfach ausführen (`Shift` + `Enter`) |
| 🛠️ | **Challenge** — hier schreibst du selbst Code |
| ✅ | Selbsttest — sagt dir sofort, ob deine Lösung stimmt |
| 💡 | **Lösung** — zum Aufklappen, wenn du nicht weiterkommst |

**Wichtig:** Führe die Zellen **von oben nach unten** aus. Spätere Zellen brauchen die
Funktionen, die du vorher schreibst.

Es sind insgesamt **5 Challenges**.

---
## 0 · Setup

▶️ Führe die beiden nächsten Zellen aus.

Die erste holt die Pakete, die zweite die Funktionen aus `helfer.py`. Gerechnet wird lokal mit
**Ollama**: `qwen3.5:0.8b` beantwortet die Fragen, `nomic-embed-text` erzeugt die Embeddings.

Die Verbindung zum Modell steht an genau einer Stelle, nämlich in `helfer.py`:

```python
BASIS_URL = "http://localhost:11434/v1"
API_KEY = "ollama"                # Ollama prüft den Key nicht
MODELL = "qwen3.5:0.8b"           # 1,0 GB, läuft auf jedem Laptop
EMBEDDING_MODELL = "nomic-embed-text"
REASONING = "none"                # qwen3.5 denkt sonst und liefert leeren content

client = OpenAI(base_url=BASIS_URL, api_key=API_KEY)
```

Das Notebook **importiert** diese Namen und definiert sie nicht noch einmal — sonst gäbe es zwei
Stellen, an denen der Modellname steht, und ein Wechsel würde an einer davon nicht greifen. Für
einen anderen Provider änderst du `BASIS_URL`, `API_KEY` und die beiden Modellnamen in
`helfer.py`. Kennt dein Provider `reasoning_effort` nicht, setzt du `REASONING = None`.

In Google Colab gibt es kein lokales Ollama. Dort brauchst du einen Provider und musst genau
diese Zeilen anpassen.

In [ ]:
# ▶️ Pakete und Diagramm-Einstellungen
import json
import re
import sys
import textwrap
from collections import Counter
from pathlib import Path

try:
    import chromadb
    from rank_bm25 import BM25Okapi
except ImportError:
    %pip install -q chromadb rank-bm25
    import chromadb
    from rank_bm25 import BM25Okapi

import matplotlib.pyplot as plt

# Einheitliche Farben für alle Diagramme in diesem Notebook
BLAU, ORANGE, TEAL, GRAU = "#2563eb", "#e8590c", "#0d9488", "#6b7280"

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAU,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e5e7eb",
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})

print("Setup fertig ✔")

▶️ Die zweite Zelle lädt `helfer.py` samt der Verbindung zum Modell, holt die vorbereiteten
Chunks und legt die Vector Database an. `helfer.embed()` merkt sich jeden berechneten Vektor in `daten/embedding_cache.json` —
deshalb ist die Collection in wenigen Sekunden gebaut.

In [ ]:
# ▶️ helfer.py finden, Chunks laden, Vector Database bauen
for kandidat in [Path.cwd(), *Path.cwd().parents]:
    if (kandidat / "helfer.py").exists():
        sys.path.insert(0, str(kandidat))
        break

import helfer
from helfer import client, frage_llm, MODELL, EMBEDDING_MODELL

chunks = helfer.lade_chunks()
fragen = helfer.lade_fragen()
sammlung = helfer.baue_chroma(chunks, neu=True)

print(f"Modell: {MODELL} · Embeddings: {EMBEDDING_MODELL}")
print(f"{len(chunks)} Chunks aus {len({c['dok_id'] for c in chunks})} Dokumenten")
print(f"{len(fragen)} Evaluationsfragen mit erwarteten Dokumenten und Antworten")
print(f"Collection {sammlung.name!r}: {sammlung.count()} Einträge unter {helfer.CHROMA_PFAD}")

---
## 1 · Zwanzig Kandidaten, ungeordnet

📖 Der Ausgangspunkt ist eine fertige Suche. Sie besteht hier aus zwei Retrievern, wie sie ein
Framework mitbringt:

* **Semantic Search** über die Chroma-Collection — die nächsten Nachbarn der Frage im Vektorraum.
* **Keyword Search** über BM25 — die Chunks mit der höchsten Wortübereinstimmung.

Jeder liefert zehn Kandidaten, beide Listen werden aneinandergehängt. Die Scores der beiden
Verfahren liegen auf verschiedenen Skalen: Kosinus-Ähnlichkeit zwischen 0 und 1, BM25 als offene
Zahl ab 0. Deshalb wird jede Liste auf ihren eigenen besten Treffer normiert — der beste Treffer
eines Retrievers bekommt 1,0, die übrigen ihren Anteil daran.

Was dabei herauskommt, ist noch kein Kontext, sondern eine Kandidatenliste. Sie ist zu lang,
enthält Chunks doppelt und hat keine Ordnung, die zur Frage passt.

In [ ]:
# ▶️ Die Suche: zwei Retriever, eine Kandidatenliste
WORT = re.compile(r"[a-z0-9äöüß]+(?:-[a-z0-9äöüß]+)*")
FELDER = ("chunk_id", "dok_id", "titel", "quelle", "position", "text")


def worte(text):
    """Zerlegt einen Text in kleingeschriebene Wörter; Kennungen bleiben zusammen."""
    return WORT.findall(text.lower())


bm25 = BM25Okapi([worte(c["text"]) for c in chunks])


def _normiert(treffer, retriever):
    """Setzt den besten Score einer Trefferliste auf 1,0 und die übrigen ins Verhältnis dazu."""
    top = max((t["score"] for t in treffer), default=0.0) or 1.0
    return [{**t, "score": t["score"] / top, "retriever": retriever} for t in treffer]


def semantische_suche(frage, n):
    """Die n nächsten Nachbarn der Frage aus der Vector Database."""
    roh = sammlung.query(query_embeddings=helfer.embed([frage]), n_results=n)
    treffer = [{"chunk_id": kennung, "dok_id": m["dok_id"], "titel": m["titel"],
                "quelle": m["quelle"], "position": m["position"], "text": text,
                "score": 1 - abstand}
               for kennung, text, m, abstand in zip(roh["ids"][0], roh["documents"][0],
                                                    roh["metadatas"][0], roh["distances"][0])]
    return _normiert(treffer, "semantic")


def keyword_suche(frage, n):
    """Die n Chunks mit dem höchsten BM25-Score."""
    scores = bm25.get_scores(worte(frage))
    beste = sorted(range(len(chunks)), key=lambda i: -scores[i])[:n]
    treffer = [{**{f: chunks[i][f] for f in FELDER}, "score": float(scores[i])} for i in beste]
    return _normiert(treffer, "bm25")


_suchcache = {}


def suche(frage, n=20):
    """Kandidaten aus beiden Retrievern, aneinandergehängt — ohne Zusammenführung.

    Die Messreihen weiter unten stellen dieselbe Frage mehrfach. Die Suche ist
    deterministisch, also wird ihr Ergebnis je Frage einmal gemerkt.
    """
    if (frage, n) not in _suchcache:
        _suchcache[(frage, n)] = semantische_suche(frage, n // 2) + keyword_suche(frage, n // 2)
    return list(_suchcache[(frage, n)])


print("suche() liefert Kandidaten aus zwei Retrievern")

In [ ]:
# ▶️ Eine Frage, zwanzig Kandidaten
PROBEFRAGE = "Welcher Workaround schützt vor CVE-2026-4410, solange kein Patch verfügbar ist?"

kandidaten = suche(PROBEFRAGE, n=20)

haeufigkeit = Counter(t["chunk_id"] for t in kandidaten)
roh_tokens = sum(helfer.zaehle_tokens(t["text"]) for t in kandidaten)

print(f"❓ {PROBEFRAGE}")
print()
print(f"{len(kandidaten)} Kandidaten, davon {len(haeufigkeit)} verschiedene Chunks "
      f"aus {len({t['dok_id'] for t in kandidaten})} Dokumenten")
print(f"{sum(n - 1 for n in haeufigkeit.values())} Kandidaten sind Wiederholungen: "
      f"{', '.join(k for k, n in haeufigkeit.items() if n > 1)}")
print(f"{roh_tokens} Tokens allein an Chunk-Text")
print()
helfer.zeige_treffer(kandidaten)

<<EINORDNUNG_1>>

---
## 2 · Die Instruktion

📖 Ein Modell, das Kontext bekommt, benutzt ihn nicht automatisch — und es benutzt nicht nur ihn.
Es hat aus dem Training eine Meinung zu fast jeder Frage, und es antwortet lieber, als dass es
schweigt. Die Instruktion ist der Teil des Prompts, der das einschränkt. Sie legt drei Dinge fest:

**Kontextbindung.** Beantwortet wird die Frage aus dem mitgelieferten Kontext, nicht aus dem
Vorwissen des Modells. Ohne diesen Satz mischt die Antwort beides, und von außen ist nicht mehr
zu erkennen, welcher Teil woher stammt.

**Der Ausweg.** Deckt der Kontext die Frage nicht, braucht das Modell eine erlaubte Antwort, die
nicht aus einer Vermutung besteht. Ein fester Satz ist dafür besser als eine Umschreibung: Er ist
im nachgelagerten Programm prüfbar.

**Belegpflicht.** Jede Aussage bekommt eine Quelle. Als Beleg dient die Chunk-ID, die im Kontext
über jedem Block steht. Damit lässt sich jede Antwort auf die Stelle zurückführen, aus der sie
kommt — und eine erfundene Aussage fällt auf, weil ihr Beleg nicht existiert oder nicht passt.

Die Prompts in diesem Kapitel sind englisch, so wie sie auf den Folien stehen. Die Antwort soll
deutsch sein; auch das gehört in die Instruktion.

In [ ]:
# ▶️ Der feste Satz für den Fall, dass der Kontext nichts hergibt
KEINE_DECKUNG = "Not covered by the provided documents."

print(f"{KEINE_DECKUNG!r}  ({helfer.zaehle_tokens(KEINE_DECKUNG)} Tokens)")

### 🛠️ Challenge 1: Die Regeln für die Antwort festlegen

Baue baue_instruktion() aus den vorbereiteten englischen Zeilen zusammen. Ergänze nur:

1. ausschließlich aus dem Kontext antworten und nicht raten,
2. jede Aussage mit einer Chunk-ID belegen,
3. bei fehlender Deckung exakt den vorgegebenen Satz ausgeben.

Rolle, Sprache und Schlussprüfung sind bereits vorgegeben. Die Regel zur fehlenden Deckung
bleibt die letzte Zeile.


In [ ]:
def baue_instruktion(keine_deckung=KEINE_DECKUNG):
    # Der Anweisungsteil des Prompts.
    zeilen = [
        "You are the retrieval assistant of the security operations center of Nordlicht Logistik SE.",
        # TODO 1: Nur den Kontext verwenden und nicht raten.
        ...,
        # TODO 2: Belegpflicht mit einer beispielhaften Chunk-ID formulieren.
        ...,
        "Write the answer in German.",
        "Before you answer, check whether the context really contains the answer.",
        # TODO 3: Exakte Antwort bei fehlender Deckung; bleibt die letzte Zeile.
        ...,
    ]
    return "\n".join(zeilen)


In [ ]:
# ✅ Selbsttest
INSTRUKTION = baue_instruktion()
klein = INSTRUKTION.lower()

assert isinstance(INSTRUKTION, str) and INSTRUKTION.strip(), "Die Instruktion ist ein nicht leerer Text"
assert "context" in klein and "only" in klein, "Es muss dastehen, dass nur der Kontext zählt"
assert KEINE_DECKUNG in INSTRUKTION, "Der Satz für den Fall ohne Deckung muss wörtlich vorkommen"
assert re.search(r"\[[a-z0-9\-]+#\d\d\]", INSTRUKTION), "Ein Beispiel für die Belegform gehört dazu"
assert "german" in klein or "deutsch" in klein, "Die Sprache der Antwort ist festgelegt"
assert helfer.zaehle_tokens(INSTRUKTION) < 250, "Die Instruktion bleibt kurz — sie kostet Budget"
assert INSTRUKTION.strip().endswith(KEINE_DECKUNG), \
    "Die Regel für den Fall ohne Deckung steht in der letzten Zeile"
assert baue_instruktion("Keine Angabe.").strip().endswith("Keine Angabe."), \
    "Der Satz ohne Deckung ist ein Parameter, kein fester Text"

print(f"✅ Challenge 1 gelöst  ({helfer.zaehle_tokens(INSTRUKTION)} Tokens)")
print()
print(INSTRUKTION)

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def baue_instruktion(keine_deckung=KEINE_DECKUNG):
    """Der Anweisungsteil des Prompts: Kontextbindung, Ausweg, Belegpflicht, Sprache."""
    return "\n".join([
        "You are the retrieval assistant of the security operations center of Nordlicht Logistik SE.",
        "Answer the question using ONLY the context below. Do not use prior knowledge, do not guess.",
        "Back every statement with a source: give the chunk id in square brackets, exactly as it",
        "appears in the header of the context block, for example [cve-2026-3224#01].",
        "Write the answer in German.",
        "Before you answer, check whether the context really contains the answer. A passage about a",
        "different subject does not count, and neither does a number that belongs to something else.",
        "If it does not contain the answer, reply with exactly this sentence and nothing else: "
        f"{keine_deckung}",
    ])
```

`ONLY` steht in Großbuchstaben, weil die Anweisung genau an dieser Stelle am häufigsten
übergangen wird. Ein Beispiel für die Belegform kostet fünf Tokens und spart die halbe
Nachbearbeitung: Ohne Beispiel schreibt das Modell mal `(Quelle: cve-2026-3224#01)`, mal
`[1]`, mal den Dateinamen.

Die Reihenfolge der Zeilen ist nicht beliebig. Die Prüfregel steht zuletzt, direkt vor Kontext
und Frage. Steht sie in der Mitte, wird sie messbar seltener befolgt — derselbe Positionseffekt,
um den es in Abschnitt 4 noch einmal geht, diesmal innerhalb der Instruktion.

</details>

📖 Ob die Instruktion wirkt, zeigt sich nicht an einer Frage, deren Antwort im Kontext steht.
Dort antwortet das Modell auch ohne sie richtig. Sie zeigt sich an einer Frage, deren Antwort
**nicht** in der Wissensbasis steht.

Die Probe fragt nach dem CVSS-Wert von `CVE-2026-9999`. Diese Kennung gibt es nicht — weder in
der Wissensbasis noch sonst wo. Die Suche liefert trotzdem zehn Chunks, weil vier echte
CVE-Advisories in der Sammlung liegen und alle vier der Frage ähnlich sehen. Sie enthalten auch
CVSS-Werte, nur eben zu anderen Schwachstellen.

▶️ Dieselbe Frage, derselbe Kontext, einmal ohne und einmal mit Instruktion.

In [ ]:
# ▶️ Dieselbe Frage, einmal ohne und einmal mit Instruktion
OHNE_INSTRUKTION = "Answer the question."
AUSSERHALB = "Welchen CVSS-Wert hat CVE-2026-9999?"

roh_kontext = "\n\n".join(t["text"] for t in suche(AUSSERHALB, n=10))


def probe(instruktion):
    """Frage und Rohkontext mit wechselnder Instruktion an das Modell."""
    return helfer.frage_llm(f"{instruktion}\n\n### Context\n{roh_kontext}\n\n"
                            f"### Question\n{AUSSERHALB}")


print(f"❓ {AUSSERHALB}")
print("   Diese CVE-Kennung existiert nicht. In der Wissensbasis steht sie nirgends.")
print(f"   Kontext: {helfer.zaehle_tokens(roh_kontext)} Tokens aus 10 Chunks, "
      f"davon {sum(1 for t in suche(AUSSERHALB, n=10) if t['dok_id'].startswith('cve-'))} aus echten Advisories")
print()
print("── ohne Instruktion " + "─" * 77)
print(textwrap.fill(probe(OHNE_INSTRUKTION), 96))
print()
print("── mit Instruktion " + "─" * 78)
print(textwrap.fill(probe(INSTRUKTION), 96))

<<EINORDNUNG_2>>

---
## 3 · Das Antwortformat

📖 Die Instruktion sagt, woraus geantwortet wird. Das Antwortformat sagt, wie die Antwort
aussieht. Beides zusammen ist der feste Teil des Prompts — er ändert sich von Frage zu Frage
nicht und geht gleich in die Budgetrechnung ein.

Drei Felder reichen für eine belegte Antwort:

| Feld | Inhalt | Wozu |
|---|---|---|
| `Antwort` | höchstens drei Sätze | die eigentliche Auskunft, kurz genug zum Lesen |
| `Belege` | Chunk-IDs, mit Komma getrennt | maschinell prüfbar: existiert die ID, passt das Dokument? |
| `Sicherheit` | `hoch`, `mittel` oder `niedrig` | eine Selbsteinschätzung, an der sich eine Nachfrage aufhängen lässt |

Die Sicherheitsangabe ist kein Messwert. Ein Sprachmodell schätzt seine eigene Zuverlässigkeit
schlecht ein und wählt fast immer die höchste Stufe. Nützlich ist sie trotzdem: Die seltenen
Fälle, in denen `niedrig` dasteht, sind gute Kandidaten für eine zweite Suche.

▶️ Das Format ist ein kurzer Text am Ende des Prompts.

In [ ]:
# ▶️ Das Antwortformat
ANTWORTFORMAT = "\n".join([
    "Use exactly this format, one field per line:",
    "Antwort: <at most three sentences>",
    "Belege: <chunk ids, comma separated>",
    "Sicherheit: hoch | mittel | niedrig",
])

print(ANTWORTFORMAT)
print()
print(f"{helfer.zaehle_tokens(ANTWORTFORMAT)} Tokens")

---
## 4 · Re-Ranking in drei Schritten

📖 Die Suche ordnet nach Ähnlichkeit zur Frage. Der Prompt braucht mehr als das: keine
Wiederholungen, eine Größe, die ins Context Window passt, und eine Reihenfolge, die zur
Aufmerksamkeit des Modells passt. Das ist die Aufgabe des Re-Rankings — drei Schritte, die
nacheinander auf die Kandidatenliste angewendet werden:

```
20 Kandidaten ──► Duplikate raus ──► ins Budget schneiden ──► Reihenfolge setzen ──► Kontext
```

Dazwischen steht noch eine vierte, triviale Kürzung: auf eine feste Höchstzahl von Chunks. Sie
braucht keine eigene Funktion, aber sie ist die Stellschraube, die in Abschnitt 8 am meisten
bewegt.

Jeder Schritt wird gemessen: Wie viele Chunks bleiben übrig, und wie viele Tokens belegen sie?

### 🛠️ Challenge 2: Doppelte Treffer entfernen

Retrieval liefert oft überlappende Chunks. Wir behalten den Treffer mit dem höheren Score und
sparen den doppelten Kontext.

1. aehnlichkeit() berechnet Jaccard: Schnittmenge geteilt durch Vereinigungsmenge.
2. entferne_duplikate() sortiert nach Score. Ein Treffer bleibt nur, wenn seine ID neu ist und
   seine Ähnlichkeit zu allen behaltenen Treffern unter der Schwelle liegt.

Das Codegerüst enthält Schleife und Fallunterscheidung. Du ergänzt Mengen und Vergleich.


In [ ]:
def aehnlichkeit(a, b):
    # Jaccard-Ähnlichkeit zweier Texte.
    # TODO 1: Beide Texte mit worte() in Mengen umwandeln.
    menge_a, menge_b = ..., ...
    if not menge_a or not menge_b:
        return 0.0
    # TODO 2: Schnittmenge durch Vereinigung teilen.
    return ...


def entferne_duplikate(treffer, schwelle=0.6):
    # Wirft gleiche oder stark überlappende Chunks weg.
    behalten = []
    for t in sorted(treffer, key=lambda t: -t["score"]):
        # TODO 3: Gleiche ID oder Ähnlichkeit ab der Schwelle erkennen.
        doppelt = ...
        if not doppelt:
            behalten.append(t)
    return behalten


In [ ]:
# ✅ Selbsttest
PROBE = [
    {"chunk_id": "a#00", "score": 0.9,
     "text": "Der Notfall-Patch wird innerhalb von 72 Stunden eingespielt."},
    {"chunk_id": "a#00", "score": 0.4,
     "text": "Der Notfall-Patch wird innerhalb von 72 Stunden eingespielt."},
    {"chunk_id": "b#00", "score": 0.7,
     "text": "Der Notfall-Patch wird innerhalb von 72 Stunden eingespielt und dokumentiert."},
    {"chunk_id": "c#00", "score": 0.5,
     "text": "Firewall-Logs liegen 90 Tage im schnellen Zugriff."},
]

assert aehnlichkeit("Token rotieren", "Token rotieren") == 1.0, "Gleicher Text: Ähnlichkeit 1,0"
assert aehnlichkeit("Token", "Patch") == 0.0, "Keine gemeinsamen Wörter: 0,0"
assert aehnlichkeit("", "Token") == 0.0, "Leerer Text: 0,0, keine Division durch null"
assert round(aehnlichkeit("a b c d", "a b c e"), 3) == 0.6, "3 gemeinsame von 5 Wörtern"

gefiltert = entferne_duplikate(PROBE)
assert [t["chunk_id"] for t in gefiltert] == ["a#00", "c#00"], \
    "a#00 doppelt, b#00 fast wortgleich mit a#00 — übrig bleiben a#00 und c#00"
assert gefiltert[0]["score"] == 0.9, "Von zwei gleichen Chunks bleibt der mit dem besseren Score"
assert len(entferne_duplikate(PROBE, schwelle=1.01)) == 3, \
    "Bei einer Schwelle über 1,0 fallen nur exakte Duplikate weg"
assert entferne_duplikate([]) == [], "Leere Liste bleibt leer"

print("✅ Challenge 2 gelöst")
for t in gefiltert:
    print(f"  {t['score']:.2f}  {t['chunk_id']}  {t['text']}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def aehnlichkeit(a, b):
    """Jaccard-Ähnlichkeit zweier Texte über ihre Wortmengen."""
    menge_a, menge_b = set(worte(a)), set(worte(b))
    if not menge_a or not menge_b:
        return 0.0
    return len(menge_a & menge_b) / len(menge_a | menge_b)


def entferne_duplikate(treffer, schwelle=0.6):
    """Behält je Chunk den besten Score und wirft stark überlappende Chunks weg."""
    behalten = []
    for t in sorted(treffer, key=lambda t: -t["score"]):
        doppelt = any(t["chunk_id"] == b["chunk_id"]
                      or aehnlichkeit(t["text"], b["text"]) >= schwelle
                      for b in behalten)
        if not doppelt:
            behalten.append(t)
    return behalten
```

Das Sortieren nach Score steht vor der Schleife, damit bei einem Duplikat immer die bessere
Fassung überlebt. Der Vergleich läuft gegen alle bereits behaltenen Treffer, nicht nur gegen den
letzten — sonst käme ein Chunk durch, der dem vorletzten gleicht.

</details>

In [ ]:
# ▶️ Was die Entdopplung an der Probefrage bewirkt
eindeutig = entferne_duplikate(kandidaten)

vorher_tokens = sum(helfer.zaehle_tokens(t["text"]) for t in kandidaten)
nachher_tokens = sum(helfer.zaehle_tokens(t["text"]) for t in eindeutig)

print(f"{len(kandidaten)} Kandidaten  →  {len(eindeutig)} Chunks")
print(f"{vorher_tokens} Tokens  →  {nachher_tokens} Tokens  "
      f"({nachher_tokens / vorher_tokens - 1:+.0%})")
print()

paare = []
for i, a in enumerate(eindeutig):
    for b in eindeutig[i + 1:]:
        paare.append((aehnlichkeit(a["text"], b["text"]), a["chunk_id"], b["chunk_id"]))
paare.sort(reverse=True)

print("Die fünf ähnlichsten Paare unter den übrig gebliebenen Chunks:")
for wert, a, b in paare[:5]:
    print(f"  {wert:.3f}  {a:<28} {b}")
print()
print("Was eine schärfere Schwelle zusätzlich wegwerfen würde:")
for schwelle in (0.6, 0.5, 0.4, 0.3):
    weniger = len(eindeutig) - len(entferne_duplikate(kandidaten, schwelle=schwelle))
    print(f"  Schwelle {schwelle:.1f}: {weniger} weitere Chunks")

<<EINORDNUNG_3>>

📖 **Das Token-Budget.** Das Context Window ist die harte Grenze: Prompt und Antwort zusammen
müssen hineinpassen. Wird sie überschritten, schneidet der Server ab — meistens vorn, also genau
dort, wo die Instruktion steht.

Das Budget für den Kontext ist deshalb nicht das ganze Fenster, sondern der Rest:

```
Budget = Context Window − Instruktion − Frage − Antwortformat − Platz für die Antwort − Reserve
```

Die Reserve fängt zwei Dinge ab: die Chat-Vorlage, die der Server um den Prompt herumlegt, und
den Unterschied zwischen dem Tokenizer, mit dem hier gezählt wird (`cl100k_base`), und dem des
Modells. Beide Zahlen sind klein, aber nicht null.

Ein Chunk kostet im Kontext mehr als seinen Text: Dazu kommen die Kopfzeile mit Chunk-ID, Titel
und Quelle und der Trenner zum nächsten Block. Das sind je nach Länge des Titels 34 bis 66
Tokens. Geschätzt wird das nicht — die Kopfzeile wird gebaut und gezählt.

In [ ]:
# ▶️ Was ein Chunk kostet und was für den Kontext übrig bleibt
CONTEXT_WINDOW = 8192      # so lädt Ollama qwen3.5:0.8b; `ollama ps` zeigt den Wert
ANTWORT_TOKENS = 600       # der Vorgabewert von helfer.frage_llm(max_tokens=...)
RESERVE = 200              # Chat-Vorlage des Servers und Unterschiede im Tokenizer
TRENNER = "\n\n---\n\n"    # steht zwischen zwei Chunks im Kontext


def kopfzeile(treffer):
    """Die Zeile, die im Kontext über einem Chunk steht: Beleg, Titel, Quelle, Position."""
    return (f"[{treffer['chunk_id']}] {treffer['titel']} · Quelle: {treffer['quelle']} "
            f"· Abschnitt {treffer['position']}")


def chunk_tokens(treffer):
    """Tokens, die ein Chunk im Kontext belegt: Kopfzeile, Text und Trenner."""
    return helfer.zaehle_tokens(f"{kopfzeile(treffer)}\n{treffer['text']}{TRENNER}")


def kontext_budget(frage, instruktion, antwortformat, context_window=CONTEXT_WINDOW):
    """Tokens, die nach den festen Teilen und dem Platz für die Antwort übrig bleiben."""
    fest = sum(helfer.zaehle_tokens(t) for t in (frage, instruktion, antwortformat))
    return context_window - fest - ANTWORT_TOKENS - RESERVE


BUDGET = kontext_budget(PROBEFRAGE, INSTRUKTION, ANTWORTFORMAT)
ANZAHL = 10                # wie viele Chunks höchstens in den Kontext kommen

print(f"Context Window      {CONTEXT_WINDOW:>6}")
for name, text in (("Instruktion", INSTRUKTION), ("Antwortformat", ANTWORTFORMAT),
                   ("Frage", PROBEFRAGE)):
    print(f"  − {name:<17}{helfer.zaehle_tokens(text):>4}")
print(f"  − {'Antwort':<17}{ANTWORT_TOKENS:>4}")
print(f"  − {'Reserve':<17}{RESERVE:>4}")
print(f"{'= Budget':<20}{BUDGET:>6} Tokens für den Kontext")
print()
print(f"Die {len(eindeutig)} Chunks der Probefrage kosten "
      f"{sum(chunk_tokens(t) for t in eindeutig)} Tokens.")
print(f"Ins Fenster passen sie damit alle. Trotzdem kommen höchstens ANZAHL = {ANZAHL} in den")
print("Kontext — warum, wird in Abschnitt 8 gemessen.")

### 🛠️ Challenge 3: Ins Budget schneiden

Schreibe `passe_ins_budget(treffer, max_tokens)`. Die Funktion geht die Treffer **nach Score
absteigend** durch und nimmt jeden auf, dessen Kosten noch ins Budget passen. Was nicht mehr
passt, wird übersprungen — die Funktion bricht nicht ab, sondern probiert die nächsten weiter.
Ein kurzer Chunk kann also noch hineinrutschen, nachdem ein langer nicht mehr gepasst hat.

Rückgabe ist die Liste der aufgenommenen Treffer in absteigender Score-Reihenfolge. Die Kosten
eines Treffers liefert `chunk_tokens()`.

*Tipp: Eine Schleife mit einem Zähler für die schon verbrauchten Tokens. `continue` überspringt
einen Treffer, ohne die Schleife zu beenden.*

In [ ]:
def passe_ins_budget(treffer, max_tokens):
    # Nimmt Treffer auf, solange das Token-Budget reicht.
    gewaehlt, verbraucht = [], 0
    for t in sorted(treffer, key=lambda t: -t["score"]):
        kosten = chunk_tokens(t)
        # TODO 1: Überspringen, wenn der Treffer nicht mehr passt.
        if ...:
            continue
        # TODO 2: Aufnehmen und Kosten addieren.
        ...
        ...
    return gewaehlt


In [ ]:
# ✅ Selbsttest
METADATEN = {"titel": "Probedokument", "quelle": "wissensbasis/probe.md", "position": 0}
BUDGET_PROBE = [
    {"chunk_id": "gross#00", "score": 0.9, "text": "Sicherheitsvorfall " * 40, **METADATEN},
    {"chunk_id": "gross#01", "score": 0.8, "text": "Sicherheitsvorfall " * 40, **METADATEN},
    {"chunk_id": "klein#00", "score": 0.1, "text": "Sicherheitsvorfall " * 2, **METADATEN},
]
kosten = [chunk_tokens(t) for t in BUDGET_PROBE]

assert passe_ins_budget(BUDGET_PROBE, 0) == [], "Ohne Budget passt nichts hinein"
assert [t["chunk_id"] for t in passe_ins_budget(BUDGET_PROBE, kosten[0])] == ["gross#00"], \
    "Genau ein Chunk passt, und zwar der mit dem besten Score"
assert [t["chunk_id"] for t in passe_ins_budget(BUDGET_PROBE, kosten[0] + kosten[1])] \
    == ["gross#00", "gross#01"], "Zwei Chunks, nach Score absteigend"
assert [t["chunk_id"] for t in passe_ins_budget(BUDGET_PROBE, kosten[0] + kosten[2])] \
    == ["gross#00", "klein#00"], \
    "Der kleine Chunk passt noch, obwohl der zweite große davor nicht mehr passt"
assert [t["chunk_id"] for t in passe_ins_budget(list(reversed(BUDGET_PROBE)), 10_000)] \
    == ["gross#00", "gross#01", "klein#00"], "Die Eingabereihenfolge spielt keine Rolle"
assert sum(chunk_tokens(t) for t in passe_ins_budget(eindeutig, 500)) <= 500, \
    "Das Budget wird nie überschritten"

im_budget = passe_ins_budget(eindeutig, BUDGET)
print("✅ Challenge 3 gelöst")
print(f"{len(eindeutig)} Chunks  →  {len(im_budget)} passen in {BUDGET} Tokens "
      f"({sum(chunk_tokens(t) for t in im_budget)} belegt)")
print()
print(f"{'Context Window':>16}{'Budget':>10}{'Chunks':>9}{'Tokens':>9}")
for fenster in (2048, 4096, 8192, 32768):
    b = kontext_budget(PROBEFRAGE, INSTRUKTION, ANTWORTFORMAT, fenster)
    gewaehlt = passe_ins_budget(eindeutig, b)
    print(f"{fenster:>16}{b:>10}{len(gewaehlt):>9}"
          f"{sum(chunk_tokens(t) for t in gewaehlt):>9}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def passe_ins_budget(treffer, max_tokens):
    """Nimmt die Treffer nach Score absteigend auf, solange das Token-Budget reicht."""
    gewaehlt, verbraucht = [], 0
    for t in sorted(treffer, key=lambda t: -t["score"]):
        kosten = chunk_tokens(t)
        if verbraucht + kosten > max_tokens:
            continue
        gewaehlt.append(t)
        verbraucht += kosten
    return gewaehlt
```

`continue` statt `break` ist eine Entscheidung mit Nebenwirkung: Sie füllt das Budget besser aus,
kann aber einen schwachen kurzen Chunk vor einen starken langen setzen. Wer das nicht will,
bricht beim ersten Chunk ab, der nicht mehr passt.

</details>

📖 **Lost in the Middle.** Die Reihenfolge im Kontext ist keine Formsache. Liu et al. haben 2024
gemessen, wie die Genauigkeit eines Modells davon abhängt, an welcher Stelle im Kontext die
benötigte Information steht: Sie ist hoch, wenn die Stelle am Anfang steht, fast ebenso hoch am
Ende — und fällt dazwischen deutlich ab. Der Verlauf sieht aus wie ein U.

```
Genauigkeit
  ▲
  │ ●                                           ●
  │      ●                                 ●
  │            ●                    ●
  │                  ●        ●
  └──────────────────────────────────────────────►  Position im Kontext
    Anfang                                  Ende
```

Die Folgerung für den Prompt: Die stärksten Treffer gehören an den Anfang und ans Ende, die
schwächeren in die Mitte. Wie stark der Effekt bei einem bestimmten Modell und einer bestimmten
Kontextlänge ausfällt und ob die Kurve wirklich symmetrisch ist, ist damit noch nicht gesagt —
das ist eine Messung, und Abschnitt 9 macht sie.

### 🛠️ Challenge 4: Nach Aufmerksamkeit ordnen

Schreibe `ordne_nach_aufmerksamkeit(treffer)`. Die Funktion sortiert zuerst nach Score
absteigend und verteilt die Treffer dann so, dass die besten außen stehen:

```
Score   0,9  0,8  0,7  0,6  0,5
        ──────────────────────►
Ausgabe 0,9  0,7  0,5  0,6  0,8
        Anfang      Mitte    Ende
```

Der beste Treffer steht vorn, der zweitbeste ganz hinten, der drittbeste an zweiter Stelle, der
viertbeste an vorletzter — und der schwächste landet in der Mitte.

*Tipp: Zwei Listen. Beim Durchlaufen der sortierten Treffer wandert jeder mit geradem Index in
die erste, jeder mit ungeradem in die zweite. Am Ende wird die zweite Liste umgedreht angehängt.*

In [ ]:
def ordne_nach_aufmerksamkeit(treffer):
    # Stärkste Treffer an Anfang und Ende.
    sortiert = sorted(treffer, key=lambda t: -t["score"])
    vorne, hinten = [], []
    for i, t in enumerate(sortiert):
        # TODO 1: Gerade Positionen nach vorne, ungerade nach hinten.
        ...
    # TODO 2: Hintere Liste umgekehrt anhängen.
    return ...


In [ ]:
# ✅ Selbsttest
ORDNUNGS_PROBE = [{"chunk_id": f"x#{i:02d}", "score": s, "text": "t"}
                  for i, s in enumerate([0.9, 0.8, 0.7, 0.6, 0.5])]
geordnet = ordne_nach_aufmerksamkeit(ORDNUNGS_PROBE)

assert [t["score"] for t in geordnet] == [0.9, 0.7, 0.5, 0.6, 0.8], "0,9 0,7 0,5 0,6 0,8"
assert geordnet[0]["score"] == 0.9, "Der beste Treffer steht vorn"
assert geordnet[-1]["score"] == 0.8, "Der zweitbeste steht ganz hinten"
assert geordnet[len(geordnet) // 2]["score"] == 0.5, "Der schwächste steht in der Mitte"
assert sorted(t["chunk_id"] for t in geordnet) == [t["chunk_id"] for t in ORDNUNGS_PROBE], \
    "Es geht kein Treffer verloren und es kommt keiner dazu"
assert [t["score"] for t in ordne_nach_aufmerksamkeit(list(reversed(ORDNUNGS_PROBE)))] \
    == [0.9, 0.7, 0.5, 0.6, 0.8], "Die Eingabereihenfolge spielt keine Rolle"
assert ordne_nach_aufmerksamkeit([]) == [], "Leere Liste bleibt leer"
assert len(ordne_nach_aufmerksamkeit(ORDNUNGS_PROBE[:1])) == 1, "Ein einzelner Treffer bleibt"

print("✅ Challenge 4 gelöst")
print("Score-Verlauf im Kontext: " + "  ".join(f"{t['score']:.1f}" for t in geordnet))

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def ordne_nach_aufmerksamkeit(treffer):
    """Die stärksten Treffer an Anfang und Ende, die schwächsten in die Mitte."""
    sortiert = sorted(treffer, key=lambda t: -t["score"])
    vorne, hinten = [], []
    for i, t in enumerate(sortiert):
        (vorne if i % 2 == 0 else hinten).append(t)
    return vorne + hinten[::-1]
```

`hinten[::-1]` dreht die zweite Hälfte um. Ohne das Umdrehen stünde der zweitbeste Treffer in der
Mitte statt am Ende — also genau dort, wo er am wenigsten gelesen wird.

</details>

In [ ]:
# ▶️ Die vier Schritte hintereinander
schritte = [("Suche", kandidaten)]
schritte.append(("Duplikate raus", entferne_duplikate(kandidaten)))
schritte.append((f"auf {ANZAHL} gekürzt", schritte[-1][1][:ANZAHL]))
schritte.append((f"ins Budget ({BUDGET})", passe_ins_budget(schritte[-1][1], BUDGET)))
schritte.append(("Reihenfolge", ordne_nach_aufmerksamkeit(schritte[-1][1])))

kopf = f"{'Schritt':<22}{'Chunks':>8}{'Tokens':>9}   Score-Verlauf"
print(kopf)
print("-" * 82)
for name, liste in schritte:
    verlauf = " ".join(f"{t['score']:.2f}" for t in liste[:10])
    if len(liste) > 10:
        verlauf += " …"
    print(f"{name:<22}{len(liste):>8}{sum(chunk_tokens(t) for t in liste):>9}   {verlauf}")

eng = kontext_budget(PROBEFRAGE, INSTRUKTION, ANTWORTFORMAT, context_window=2048)
knapp = passe_ins_budget(entferne_duplikate(kandidaten)[:ANZAHL], eng)
print()
print(f"Mit einem Context Window von 2048 wäre das Budget {eng} Tokens — dann kämen "
      f"{len(knapp)} statt {len(schritte[-1][1])} Chunks in den Prompt.")

<<EINORDNUNG_4>>

---
## 5 · Der Kontextteil und der fertige Prompt

📖 Der Kontext ist nicht einfach die Aneinanderreihung der Chunk-Texte. Jeder Chunk bekommt eine
Kopfzeile, und die trägt vier Angaben:

```
[cve-2026-4410#02] CVE-2026-4410 — Denial of Service … · Quelle: wissensbasis/cve-2026-4410.md · Abschnitt 2
```

Die **Chunk-ID** in eckigen Klammern ist der Beleg, auf den sich die Antwort beruft. Sie steht am
Anfang der Zeile, weil das Modell sie von dort am zuverlässigsten übernimmt. **Titel** und
**Quelle** ordnen die Textstelle einem Dokument zu — ein Chunk aus der Mitte eines Runbooks
enthält den Namen des Runbooks sonst nirgends. Die **Position** sagt, aus welchem Abschnitt des
Dokuments die Stelle stammt.

Zwischen den Blöcken steht ein Trenner. Ohne ihn laufen zwei Chunks aus verschiedenen Dokumenten
zu einem Absatz zusammen, und das Modell zitiert die Quelle des falschen.

Der fertige Prompt setzt die vier Teile in dieser Reihenfolge zusammen:

```
Instruktion            was gilt
### Context            die Chunks mit ihren Kopfzeilen
### Question           die Frage
### Answer             das Antwortformat
```

Die Frage steht hinter dem Kontext, nicht davor. Damit ist sie das Letzte, was das Modell vor dem
Schreiben liest.

### 🛠️ Challenge 5: Den finalen Augmented Prompt bauen

Die vorbereiteten Teile werden nur noch zusammengesetzt:

- baue_kontext() erzeugt je Treffer eine Kopfzeile und darunter den Text.
- baue_prompt() setzt Instruktion, Context, Question und Answer in dieser Reihenfolge zusammen.

Hier geht es ausschließlich um eine eindeutige Prompt-Struktur und zitierbare Chunk-IDs.


In [ ]:
def baue_kontext(treffer):
    # Kopfzeile und Text je Chunk.
    # TODO 1: Je Treffer einen Block bauen.
    bloecke = ...
    # TODO 2: Blöcke mit TRENNER verbinden.
    return ...


def baue_prompt(frage, treffer, instruktion, antwortformat):
    # Instruktion, Kontext, Frage und Antwortformat.
    kontext = baue_kontext(treffer)
    # TODO 3: Teile mit den vorgegebenen Überschriften zusammensetzen.
    return ...


In [ ]:
# ✅ Selbsttest
final = ordne_nach_aufmerksamkeit(
    passe_ins_budget(entferne_duplikate(kandidaten)[:ANZAHL], BUDGET))
kontext = baue_kontext(final)
prompt = baue_prompt(PROBEFRAGE, final, INSTRUKTION, ANTWORTFORMAT)

assert baue_kontext([]) == "", "Ohne Treffer ist der Kontext leer"
assert all(t["chunk_id"] in kontext for t in final), "Jede Chunk-ID steht im Kontext"
assert all(t["quelle"] in kontext for t in final), "Jede Quelle steht im Kontext"
assert all(t["titel"] in kontext for t in final), "Jeder Titel steht im Kontext"
assert all(t["text"] in kontext for t in final), "Kein Chunk-Text wird abgeschnitten"
assert kontext.count(TRENNER) == len(final) - 1, "Zwischen zwei Blöcken steht ein Trenner"
assert abs(helfer.zaehle_tokens(kontext) - sum(chunk_tokens(t) for t in final)) <= 10, \
    "Was der Kontext kostet, hat die Budgetrechnung vorher gewusst"
assert kontext.index(final[0]["text"]) < kontext.index(final[-1]["text"]), \
    "Die Reihenfolge aus dem Re-Ranking bleibt erhalten"

assert prompt.index(INSTRUKTION) < prompt.index("### Context") < prompt.index("### Question"), \
    "Erst die Instruktion, dann der Kontext, dann die Frage"
assert prompt.index("### Question") < prompt.index("### Answer"), "Das Antwortformat steht zuletzt"
assert PROBEFRAGE in prompt and ANTWORTFORMAT in prompt, "Frage und Antwortformat fehlen nicht"
assert kontext in prompt, "Der Kontext steht unverändert im Prompt"
assert helfer.zaehle_tokens(prompt) + ANTWORT_TOKENS + RESERVE <= CONTEXT_WINDOW, \
    "Prompt und Antwort zusammen bleiben im Context Window"

print("✅ Challenge 5 gelöst")
print(f"{len(final)} Chunks, Kontext {helfer.zaehle_tokens(kontext)} Tokens, "
      f"Prompt {helfer.zaehle_tokens(prompt)} Tokens")
print()
print(kontext[:600] + " …")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def baue_kontext(treffer):
    """Der Kontextteil: je Chunk die Kopfzeile mit Beleg und Quelle, darunter der Text."""
    bloecke = [f"{kopfzeile(t)}\n{t['text']}" for t in treffer]
    return TRENNER.join(bloecke)


def baue_prompt(frage, treffer, instruktion, antwortformat):
    """Instruktion, Kontext, Frage und Antwortformat in einem Prompt."""
    return (f"{instruktion}\n\n"
            f"### Context\n{baue_kontext(treffer)}\n\n"
            f"### Question\n{frage}\n\n"
            f"### Answer\n{antwortformat}")
```

`TRENNER.join(...)` setzt den Trenner nur zwischen die Blöcke, nicht davor und nicht dahinter.
Eine Schleife, die den Trenner an jeden Block anhängt, produziert am Ende einen Trenner ohne
Inhalt — und das Modell zitiert dann gelegentlich einen leeren Block.

`kopfzeile()` steht schon in der Budgetrechnung. Dieselbe Funktion an beiden Stellen zu benutzen
ist kein Sparen an Zeilen: Sobald die Kopfzeile hier anders aussieht als dort, rechnet das Budget
mit einem Kontext, den es nicht gibt.

</details>

---
## 6 · Was beim Modell ankommt

📖 An dieser Stelle ist der Prompt fertig. Bevor er in einer Funktion verschwindet, steht er
einmal vollständig da: derselbe Text, den der Server sieht.

▶️ Die nächste Zelle druckt ihn im Klartext, mit Zeilennummern für die Kopfzeilen.

In [ ]:
# ▶️ Der vollständige Prompt im Klartext
print(prompt)

In [ ]:
# ▶️ Und die Antwort dazu
antwort = helfer.frage_llm(prompt)

print(antwort)
print()
print(f"Prompt {helfer.zaehle_tokens(prompt)} Tokens · Antwort {helfer.zaehle_tokens(antwort)} Tokens")

<<EINORDNUNG_6>>

---
## 7 · Die ganze Kette über zehn Fragen

📖 Die Einzelteile stehen. Zusammengesetzt ergeben sie eine Funktion, die eine Frage
entgegennimmt und eine belegte Antwort zurückgibt:

```
Frage ──► suche ──► entferne_duplikate ──► passe_ins_budget ──► Reihenfolge ──► baue_prompt ──► frage_llm
```

Gemessen wird an den zehn Fragen aus `daten/fragen.json`. Zu jeder gehören die Dokumente, in
denen die Antwort steht, und eine Musterantwort. Daraus lassen sich drei Kennzahlen bilden:

* **Deckung** — welcher Anteil der Kernbegriffe der Musterantwort in der Antwort des Modells
  vorkommt. Kernbegriffe sind Wörter mit einer Ziffer (Zahlen, Versionen, Kennungen) und Wörter
  ab sechs Zeichen. Das Maß ist grob: Es zählt Zeichenketten, nicht Bedeutung, und 100 Prozent
  erreicht nur, wer die Musterantwort wörtlich nachbaut. Für den Vergleich zweier Einstellungen
  reicht es, weil beide Seiten denselben Maßstab bekommen.
* **Quelle** — ob mindestens ein Beleg in der Antwort zu einem der erwarteten Dokumente gehört.
* **Format** — ob die drei Felder `Antwort`, `Belege` und `Sicherheit` dastehen und die
  Sicherheitsstufe eine der drei erlaubten ist.

In [ ]:
# ▶️ Die Kette als eine Funktion
ORDNUNGEN = {
    "score": lambda t: sorted(t, key=lambda x: -x["score"]),
    "umgekehrt": lambda t: sorted(t, key=lambda x: x["score"]),
    "aufmerksamkeit": ordne_nach_aufmerksamkeit,
}


def hole_kontext(frage, anzahl=ANZAHL, budget=None, ordnung="aufmerksamkeit"):
    """Suche und Re-Ranking: von der Frage zu den Chunks, die in den Prompt kommen."""
    if budget is None:
        budget = kontext_budget(frage, INSTRUKTION, ANTWORTFORMAT)
    treffer = entferne_duplikate(suche(frage, n=20))
    treffer = sorted(treffer, key=lambda t: -t["score"])[:anzahl]
    return ORDNUNGEN[ordnung](passe_ins_budget(treffer, budget))


def antworte(frage, anzahl=ANZAHL, budget=None, ordnung="aufmerksamkeit"):
    """Die ganze Kette: Suche, Re-Ranking, Prompt, Antwort."""
    treffer = hole_kontext(frage, anzahl=anzahl, budget=budget, ordnung=ordnung)
    prompt = baue_prompt(frage, treffer, INSTRUKTION, ANTWORTFORMAT)
    return {"frage": frage, "treffer": treffer, "prompt": prompt,
            "antwort": helfer.frage_llm(prompt)}


print("antworte() steht")

In [ ]:
# ▶️ Die drei Kennzahlen
FUELLWOERTER = {"eines", "einer", "einem", "sowie", "werden", "wurde", "wurden", "insgesamt",
                "danach", "durch", "nicht", "dieser", "diese", "dieses", "beziehungsweise"}
BELEG_MUSTER = re.compile(r"\[?([a-z0-9][a-z0-9\-]*#\d{2})\]?")


def _zahlen_vereinheitlicht(text):
    """Macht aus 9,8 und 9.8 dieselbe Zeichenfolge."""
    return re.sub(r"(?<=\d)[,.](?=\d)", ".", text.lower())


def kernbegriffe(text):
    """Wörter mit Ziffer und Wörter ab sechs Zeichen — der prüfbare Kern einer Antwort."""
    return {w for w in worte(_zahlen_vereinheitlicht(text))
            if (any(z.isdigit() for z in w) or len(w) >= 6) and w not in FUELLWOERTER}


def deckung(antwort, musterantwort):
    """Anteil der Kernbegriffe der Musterantwort, die in der Antwort vorkommen."""
    soll = kernbegriffe(musterantwort)
    ist = _zahlen_vereinheitlicht(antwort)
    return sum(1 for w in soll if w in ist) / len(soll) if soll else 0.0


def belegte_dok_ids(antwort):
    """Die Dokumente, auf die sich die Belege in der Antwort beziehen."""
    return {kennung.split("#")[0] for kennung in BELEG_MUSTER.findall(antwort.lower())}


def format_erfuellt(antwort):
    """Stehen die drei Felder da, und ist die Sicherheitsstufe eine der erlaubten?"""
    return (bool(re.search(r"^Antwort:", antwort, re.M))
            and bool(re.search(r"^Belege:", antwort, re.M))
            and bool(re.search(r"^Sicherheit:\s*(hoch|mittel|niedrig)\s*$", antwort, re.M | re.I)))


def bewerte(lauf, frage):
    """Die drei Kennzahlen für eine beantwortete Frage."""
    return {"deckung": deckung(lauf["antwort"], frage["erwartete_antwort"]),
            "quelle": bool(belegte_dok_ids(lauf["antwort"]) & set(frage["erwartete_dok_ids"])),
            "format": format_erfuellt(lauf["antwort"]),
            "chunks": len(lauf["treffer"]),
            "tokens": helfer.zaehle_tokens(lauf["prompt"])}


print("Kennzahlen: deckung, quelle, format")

In [ ]:
# ▶️ Alle zehn Fragen durch die Kette
laeufe = [antworte(f["frage"]) for f in fragen]
noten = [bewerte(lauf, f) for lauf, f in zip(laeufe, fragen)]

kopf = f"{'Frage':<52}{'Chunks':>7}{'Tokens':>8}{'Deckung':>9}{'Quelle':>8}{'Format':>8}"
print(kopf)
print("-" * len(kopf))
for f, note in zip(fragen, noten):
    print(f"{f['frage'][:50]:<52}{note['chunks']:>7}{note['tokens']:>8}"
          f"{note['deckung']:>9.0%}{'✔' if note['quelle'] else '✗':>8}"
          f"{'✔' if note['format'] else '✗':>8}")

print("-" * len(kopf))
print(f"{'Mittel':<52}{sum(n['chunks'] for n in noten) / len(noten):>7.1f}"
      f"{sum(n['tokens'] for n in noten) // len(noten):>8}"
      f"{sum(n['deckung'] for n in noten) / len(noten):>9.0%}"
      f"{sum(n['quelle'] for n in noten) / len(noten):>8.0%}"
      f"{sum(n['format'] for n in noten) / len(noten):>8.0%}")

In [ ]:
# ▶️ Zwei Antworten im Wortlaut
for lauf, f in list(zip(laeufe, fragen))[:2]:
    print(f"❓ {f['frage']}")
    print(f"   erwartet: {f['erwartete_antwort']}")
    print()
    print(textwrap.indent(lauf["antwort"], "   "))
    print()

<<EINORDNUNG_7>>

▶️ Und die Gegenprobe: drei Fragen, deren Antwort nicht in der Wissensbasis steht. Erwartet wird
jetzt keine Auskunft, sondern der Satz aus der Instruktion.

In [ ]:
# ▶️ Fragen ohne Deckung in der Wissensbasis
AUSSERHALB_FRAGEN = [
    "Welchen CVSS-Wert hat CVE-2026-9999?",
    "Wie viele Urlaubstage stehen einer Mitarbeiterin der Nordlicht Logistik SE im Jahr zu?",
    "Welches Passwort hat das Konto adm-backup?",
    "Innerhalb welcher Frist muss ein Datenschutzvorfall an die Aufsichtsbehörde gemeldet werden?",
]

abgelehnt = 0
for frage in AUSSERHALB_FRAGEN:
    lauf = antworte(frage)
    verweigert = KEINE_DECKUNG.lower() in lauf["antwort"].lower()
    abgelehnt += verweigert
    print(f"{'✔' if verweigert else '✗'} {frage}")
    print(textwrap.indent(textwrap.fill(lauf["antwort"], 92), "     "))
    print()

print(f"{abgelehnt} von {len(AUSSERHALB_FRAGEN)} Fragen ohne Deckung abgelehnt")

<<EINORDNUNG_7B>>

---
## 8 · Einstellungen messen

📖 Drei Stellschrauben sind übrig: die Zahl der Chunks im Kontext, die Reihenfolge und das
Token-Budget. Über jede lässt sich streiten; gemessen wird sie in wenigen Minuten.

Jede Einstellung läuft über dieselben zehn Fragen und bekommt dieselben drei Kennzahlen. Zehn
Fragen sind eine kleine Menge — ein Unterschied von einem Zehntel entspricht einer einzigen
Frage. Die Zahlen zeigen Größenordnungen, keine Nachkommastellen.

In [ ]:
# ▶️ Eine Einstellung, zehn Fragen, drei Kennzahlen
_gemessen = {}


def messe(anzahl=ANZAHL, budget=None, ordnung="aufmerksamkeit"):
    """Dieselbe Auswertung über alle zehn Fragen, für eine Einstellung."""
    schluessel = (anzahl, budget, ordnung)
    if schluessel not in _gemessen:
        noten = [bewerte(antworte(f["frage"], anzahl=anzahl, budget=budget, ordnung=ordnung), f)
                 for f in fragen]
        _gemessen[schluessel] = {
            feld: sum(n[feld] for n in noten) / len(noten)
            for feld in ("deckung", "quelle", "format", "chunks", "tokens")}
    return _gemessen[schluessel]


def zeige_reihe(titel, spalte, zeilen):
    """Druckt eine Messreihe als Tabelle."""
    kopf = f"{spalte:<20}{'Chunks':>8}{'Tokens':>8}{'Deckung':>9}{'Quelle':>8}{'Format':>8}"
    print(titel)
    print(kopf)
    print("-" * len(kopf))
    for name, werte in zeilen:
        print(f"{name:<20}{werte['chunks']:>8.1f}{werte['tokens']:>8.0f}"
              f"{werte['deckung']:>9.0%}{werte['quelle']:>8.0%}{werte['format']:>8.0%}")
    print()


print("messe() steht")

In [ ]:
# ▶️ Reihe 1: Wie viele Chunks?
reihe_chunks = [(f"{k} Chunks", messe(anzahl=k)) for k in (3, 5, 10, 20)]
zeige_reihe("Zahl der Chunks (Reihenfolge nach Aufmerksamkeit, volles Budget)",
            "Einstellung", reihe_chunks)

In [ ]:
# ▶️ Reihe 2: In welcher Reihenfolge?
reihe_ordnung = [(name, messe(anzahl=ANZAHL, ordnung=name))
                 for name in ("score", "umgekehrt", "aufmerksamkeit")]
zeige_reihe("Reihenfolge im Kontext (10 Chunks)", "Ordnung", reihe_ordnung)

In [ ]:
# ▶️ Reihe 3: Wie groß darf der Kontext werden?
reihe_budget = [(f"{b} Tokens", messe(anzahl=20, budget=b)) for b in (400, 800, 1600)]
reihe_budget.append(("volles Budget", messe(anzahl=20)))
zeige_reihe("Token-Budget (20 Kandidaten, Reihenfolge nach Aufmerksamkeit)",
            "Budget", reihe_budget)

In [ ]:
# ▶️ Die drei Reihen im Diagramm
reihen = [("Zahl der Chunks", reihe_chunks), ("Reihenfolge", reihe_ordnung),
          ("Token-Budget", reihe_budget)]

fig, achsen = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for achse, (titel, zeilen) in zip(achsen, reihen):
    namen = [n for n, _ in zeilen]
    achse.bar(namen, [w["deckung"] for _, w in zeilen], color=BLAU, width=0.55)
    achse.set_title(titel)
    achse.set_ylim(0, 1)
    achse.tick_params(axis="x", labelrotation=30)
achsen[0].set_ylabel("Deckung der Musterantwort")
plt.tight_layout()
plt.show()

<<EINORDNUNG_8>>

---
## 9 · Lost in the Middle zum Anfassen

📖 Die Messung oben vergleicht drei Reihenfolgen einer vollständigen Trefferliste. Sie kann den
Positionseffekt nur schwach zeigen, weil in einer guten Trefferliste mehrere Chunks zur Antwort
beitragen — fällt einer aus der Aufmerksamkeit, springt ein anderer ein.

Der saubere Versuch trennt das. Zu jeder Frage gibt es genau **einen** Chunk, der die Antwort
enthält. Dazu kommen Füll-Chunks aus allen anderen Dokumenten, so viele, bis das Token-Budget
ausgeschöpft ist. Der Chunk mit der Antwort wandert durch drei Positionen: an den Anfang, in die
Mitte, ans Ende. Alles andere bleibt gleich — dieselben Füll-Chunks, dieselbe Frage, dieselbe
Instruktion.

Der Kontext wird hier absichtlich bis an die Grenze gefüllt, denn der Effekt wächst mit der
Länge des Kontexts — bei den zehn Chunks aus Abschnitt 8 bleibt er im Rauschen. Gemessen wird,
ob die Antwort die Angabe enthält und ob sie den Chunk zitiert, in dem sie steht.

In [ ]:
# ▶️ Der Chunk mit der Antwort, dazu Füll-Chunks bis ans Budget
def gold_chunk(frage):
    """Der Chunk aus einem erwarteten Dokument, der die Musterantwort am besten abdeckt."""
    aus_dokument = [c for c in chunks if c["dok_id"] in frage["erwartete_dok_ids"]]
    return max(aus_dokument, key=lambda c: deckung(c["text"], frage["erwartete_antwort"]))


def fuell_chunks(frage):
    """Chunks aus allen anderen Dokumenten, so viele wie neben dem Gold-Chunk ins Budget passen."""
    budget = kontext_budget(frage["frage"], INSTRUKTION, ANTWORTFORMAT)
    andere = [t for t in entferne_duplikate(suche(frage["frage"], n=20))
              if t["dok_id"] not in frage["erwartete_dok_ids"]]
    schon_da = {t["chunk_id"] for t in andere}
    andere += [{**c, "score": 0.0} for c in chunks
               if c["dok_id"] not in frage["erwartete_dok_ids"] and c["chunk_id"] not in schon_da]

    gewaehlt, verbraucht = [], chunk_tokens(gold_chunk(frage))
    for t in andere:
        if verbraucht + chunk_tokens(t) > budget:
            continue
        gewaehlt.append(t)
        verbraucht += chunk_tokens(t)
    return gewaehlt


def stelle_gold_ein(gold, fuell, wo):
    """Setzt den Chunk mit der Antwort an den Anfang, in die Mitte oder ans Ende."""
    index = {"Anfang": 0, "Mitte": len(fuell) // 2, "Ende": len(fuell)}[wo]
    return fuell[:index] + [gold] + fuell[index:]


print(f"{'Frage':<52}{'Chunk mit der Antwort':<32}{'Deckung':>8}{'Füller':>8}")
print("-" * 100)
for f in fragen:
    gold = gold_chunk(f)
    print(f"{f['frage'][:50]:<52}{gold['chunk_id']:<32}"
          f"{deckung(gold['text'], f['erwartete_antwort']):>8.0%}{len(fuell_chunks(f)):>8}")

beispiel = stelle_gold_ein({**gold_chunk(fragen[0]), "score": 1.0}, fuell_chunks(fragen[0]), "Mitte")
print()
print(f"Beispiel: {len(beispiel)} Chunks, "
      f"{helfer.zaehle_tokens(baue_kontext(beispiel))} Tokens Kontext")

In [ ]:
# ▶️ Derselbe Kontext, drei Positionen
POSITIONEN = ["Anfang", "Mitte", "Ende"]
positionstest = {}

for wo in POSITIONEN:
    werte, getroffen, zitiert, groessen = [], [], [], []
    for f in fragen:
        liste = stelle_gold_ein({**gold_chunk(f), "score": 1.0}, fuell_chunks(f), wo)
        antwort_text = helfer.frage_llm(baue_prompt(f["frage"], liste, INSTRUKTION, ANTWORTFORMAT))
        werte.append(deckung(antwort_text, f["erwartete_antwort"]))
        getroffen.append(werte[-1] >= 0.5)
        zitiert.append(gold_chunk(f)["chunk_id"].lower() in antwort_text.lower())
        groessen.append(len(liste))
    positionstest[wo] = {"deckung": sum(werte) / len(werte),
                         "getroffen": sum(getroffen) / len(getroffen),
                         "zitiert": sum(zitiert) / len(zitiert),
                         "chunks": sum(groessen) / len(groessen)}

kopf = (f"{'Position im Kontext':<22}{'Chunks':>8}{'Deckung':>10}"
        f"{'Deckung ≥ 50 %':>16}{'Chunk zitiert':>16}")
print(kopf)
print("-" * len(kopf))
for wo in POSITIONEN:
    w = positionstest[wo]
    print(f"{wo:<22}{w['chunks']:>8.1f}{w['deckung']:>10.0%}"
          f"{w['getroffen']:>16.0%}{w['zitiert']:>16.0%}")

In [ ]:
# ▶️ Das Ergebnis im Diagramm
stellen = range(len(POSITIONEN))
breite = 0.38

plt.figure(figsize=(8, 4.5))
plt.bar([s - breite / 2 for s in stellen], [positionstest[n]["deckung"] for n in POSITIONEN],
        breite, label="Deckung der Musterantwort", color=BLAU)
plt.bar([s + breite / 2 for s in stellen], [positionstest[n]["getroffen"] for n in POSITIONEN],
        breite, label="Fragen mit Deckung ab 50 %", color=ORANGE)
plt.xticks(list(stellen), [f"{n}\n({positionstest[n]['chunks']:.0f} Chunks)" for n in POSITIONEN])
plt.ylim(0, 1)
plt.ylabel("Anteil über zehn Fragen")
plt.title("Wo der entscheidende Chunk steht, entscheidet mit")
plt.legend()
plt.show()

<<EINORDNUNG_9>>

In [ ]:
# ▶️ Ergebnisse sichern
ergebnisse = {
    "instruktion": INSTRUKTION,
    "antwortformat": ANTWORTFORMAT,
    "keine_deckung": KEINE_DECKUNG,
    "budget": {"context_window": CONTEXT_WINDOW, "antwort_tokens": ANTWORT_TOKENS,
               "reserve": RESERVE},
    "grundeinstellung": {"anzahl": ANZAHL, "ordnung": "aufmerksamkeit"},
    "fragen": [
        {"frage": f["frage"],
         "erwartete_dok_ids": f["erwartete_dok_ids"],
         "chunk_ids": [t["chunk_id"] for t in lauf["treffer"]],
         "antwort": lauf["antwort"],
         **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in note.items()}}
        for f, lauf, note in zip(fragen, laeufe, noten)
    ],
    "reihen": {
        "chunks": {n: {k: round(v, 4) for k, v in w.items()} for n, w in reihe_chunks},
        "ordnung": {n: {k: round(v, 4) for k, v in w.items()} for n, w in reihe_ordnung},
        "budget": {n: {k: round(v, 4) for k, v in w.items()} for n, w in reihe_budget},
    },
    "position": {n: {k: round(v, 4) for k, v in w.items()} for n, w in positionstest.items()},
}

pfad = helfer.DATEN / "04_augmented_prompt.json"
pfad.write_text(json.dumps(ergebnisse, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")

print(f"{pfad.name} geschrieben: {len(ergebnisse['fragen'])} Fragen, "
      f"{sum(len(r) for r in ergebnisse['reihen'].values())} Messreihen-Zeilen")

<<ABSCHLUSS>>

---
### 🔬 Bonus — ohne Lösung

**1. Cross-Encoder als Re-Ranking.** Die Suche bettet Frage und Chunk **getrennt** ein und
vergleicht zwei Vektoren; was in der Frage steht, beeinflusst die Darstellung des Chunks nicht.
Ein Cross-Encoder liest beides **zusammen** in einem Durchlauf und gibt eine einzelne
Relevanzzahl aus — dadurch genauer, aber zu teuer für die ganze Sammlung. Deshalb steht er hinter
der Suche: Der Retriever holt zwanzig Kandidaten, der Cross-Encoder sortiert sie neu.

Zum Ausprobieren: `sentence-transformers` bringt mit
`CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")` ein kleines Modell mit. Es bewertet Paare
`(frage, chunk_text)`. Setze seine Ausgabe als `score` in die Trefferliste und lasse den Rest der
Kette unverändert. Miss dieselben drei Kennzahlen und die Zeit je Frage.

**2. Re-Ranking durch das Modell selbst.** Statt eines zweiten Modells sortiert das
Antwortmodell. Das kostet einen zusätzlichen Aufruf je Frage und braucht keine neue Abhängigkeit:

```python
def llm_reranking(frage, treffer, behalten=5):
    """Lässt das Modell die Kandidaten nach Relevanz sortieren."""
    liste = "\n\n".join(f"[{i}] {' '.join(t['text'].split())[:300]}"
                        for i, t in enumerate(treffer))
    prompt = (
        "Rank the passages by how well they answer the question.\n"
        f"Return only the {behalten} best indices as a JSON list, best first, "
        "for example [3, 0, 7, 1, 5]. No other text.\n\n"
        f"### Question\n{frage}\n\n### Passages\n{liste}\n\n### Ranking\n"
    )
    roh = helfer.frage_llm(prompt, max_tokens=60)
    treffer_im_text = re.search(r"\[[\d,\s]*\]", roh)
    if not treffer_im_text:
        return treffer[:behalten]
    reihenfolge = [i for i in json.loads(treffer_im_text.group(0)) if 0 <= i < len(treffer)]
    return [treffer[i] for i in reihenfolge[:behalten]]
```

Drei Fragen dazu: Hält sich `qwen3.5:0.8b` an das Ausgabeformat, oder greift der Ausweg über
`treffer[:behalten]` oft? Ändert sich die Deckung gegenüber der Sortierung nach Score? Und wie
viel Zeit kostet der zusätzliche Aufruf im Verhältnis zur Antwort selbst?

**3. Reciprocal Rank Fusion.** Die Suche in diesem Notebook hängt zwei Trefferlisten aneinander
und normiert jede auf ihren besten Treffer. Das ist die einfachste Form der Zusammenführung und
hat eine bekannte Schwäche: Ein Retriever mit einem knappen Feld von Scores wird gegenüber einem
mit weit gespreizten Scores benachteiligt. Reciprocal Rank Fusion rechnet stattdessen mit den
Rängen — jeder Treffer bekommt `1 / (k + rang)`, die Beiträge beider Listen werden addiert.
Ausgeführt und gemessen wird das im Bonus-Notebook `bonus_hybrid_search`.